In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

True

In [2]:
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

In [3]:
from agents import Agent, Runner, function_tool  

import asyncio   
import gradio as gr  
import tempfile
import os as _os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

c:\Users\USER\Desktop\code\AI Agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
researcher = Agent(
    name="Researcher",
    instructions=""" You are a thorough researcher.
      When given a topic:
    1. Break it down into 3-5 key subtopics
    2. Research each subtopic with facts, stats, and examples
    3. Compile findings in a structured format
    4. Identify key insights and trends

    Be thorough but concise. Focus on facts and data.
    Always organize your findings with clear headings.""",
    model="gpt-4o-mini"  
)

In [5]:
writer = Agent(
    name="ReportWriter",
    instructions="""You write professional, well-structured reports.

    Structure your reports as:
    1. Executive Summary (2-3 sentences overview)
    2. Key Findings (bullet points of most important discoveries)
    3. Detailed Analysis (deep dive into each area)
    4. Recommendations (actionable next steps)
    5. Conclusion (wrap-up)

    Use clear, professional language. Include data points where relevant.
    Use markdown formatting for readability.""",
    model="gpt-4o-mini"
)

In [6]:
editor = Agent(
    name="Editor",
    instructions="""You are a meticulous editor.

    Review and improve content for:
    - Grammar and spelling errors
    - Clarity and logical flow
    - Factual consistency
    - Professional tone
    - Readability

    Return the COMPLETE improved version with all your edits applied.
    Do NOT return comments — return the final polished text.""",
    model="gpt-4o-mini"
)

In [7]:
@function_tool
async def research(topic: str) -> str:
    """
    Conduct thorough research on a topic.
    The researcher agent will break down the topic and gather detailed findings.

    Args:
        topic: The topic to research
    """
    # Runner.run() sends the message to the researcher agent and returns its output
    result = await Runner.run(researcher, f"Research this topic comprehensively: {topic}")
    return result.final_output  # .final_output is the agent's text response



In [8]:
@function_tool
async def write_report(research_findings: str, topic: str) -> str:
    """
    Write a professional report based on research findings.
    The writer agent will structure findings into a polished report.

    Args:
        research_findings: The raw research data to base the report on
        topic: The main topic/title for the report
    """
    
    prompt = f"Write a professional report about '{topic}' based on this research:\n\n{research_findings}"
    result = await Runner.run(writer, prompt)
    return result.final_output

In [9]:
@function_tool
async def edit_report(report: str) -> str:
    """
    Edit and polish a report for grammar, clarity, and professionalism.
    The editor agent will review and return an improved version.

    Args:
        report: The full report text to edit and improve
    """
    result = await Runner.run(editor, f"Edit and improve this report:\n\n{report}")
    return result.final_output

In [10]:
report_generator = Agent(
    name="ReportGenerator",
    instructions="""You manage the full report generation pipeline.

    When the user asks for a report:
    1. First, use the 'research' tool to gather information on the topic
    2. Then, use 'write_report' tool to create a structured report from the research
    3. Finally, use 'edit_report' tool to polish and finalize the report

    IMPORTANT:
    - Always follow all 3 steps in order
    - Pass the FULL research output to the writer
    - Pass the FULL draft report to the editor
    - Present the final edited report to the user
    - Do NOT add your own commentary after the final report""",
    model="gpt-4o-mini",
    tools=[research, write_report, edit_report]  
)

In [11]:
async def generate_report(topic: str) -> str:
    """
    Main function: takes a topic string, runs the full agent pipeline,
    returns the final report as a string.
    """
    if not topic.strip():                          
        return "Please enter a topic to research."

    try:
       
        runner = await Runner.run(
            report_generator,                    
            f"Generate a comprehensive report about: {topic}", 
            max_turns=15                            
        )
        return runner.final_output                  
    except Exception as e:
        return f"Error generating report: {str(e)}"

In [12]:
def run_report(topic):
    """
    Gradio calls this synchronous function.
    We use asyncio.run() to bridge sync Gradio → async agent code.
    """
    return asyncio.run(generate_report(topic))


In [13]:
import tempfile
import re
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

def markdown_to_pdf(report_text: str) -> str:
    if not report_text or report_text.startswith("Please enter") or report_text.startswith("Error"):
        return None

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
    tmp_path = tmp.name
    tmp.close()

    doc = SimpleDocTemplate(tmp_path, pagesize=letter,
        rightMargin=72, leftMargin=72, topMargin=72, bottomMargin=72)

    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name='CustomHeading1', parent=styles['Heading1'],
        fontSize=18, spaceAfter=12, spaceBefore=20))
    styles.add(ParagraphStyle(name='CustomHeading2', parent=styles['Heading2'],
        fontSize=14, spaceAfter=8, spaceBefore=14))
    styles.add(ParagraphStyle(name='CustomBody', parent=styles['Normal'],
        fontSize=11, leading=16, spaceAfter=8))
    styles.add(ParagraphStyle(name='BulletStyle', parent=styles['Normal'],
        fontSize=11, leading=16, leftIndent=20, spaceAfter=4, bulletIndent=10))

    def clean(text):
        """Convert markdown bold to reportlab bold AND escape bad XML."""
        text = text.replace('&', '&amp;')          # Escape ampersands
        text = text.replace('<', '&lt;').replace('>', '&gt;')  # Escape angle brackets
        text = re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', text)   # **bold** → <b>bold</b>
        text = re.sub(r'__(.+?)__', r'<b>\1</b>', text)       # __bold__ → <b>bold</b>
        text = re.sub(r'\*(.+?)\*', r'<i>\1</i>', text)       # *italic* → <i>italic</i>
        return text

    story = []

    for line in report_text.split('\n'):
        stripped = line.strip()
        if not stripped:
            story.append(Spacer(1, 6))
        elif stripped.startswith('# '):
            story.append(Paragraph(clean(stripped[2:]), styles['CustomHeading1']))
        elif stripped.startswith('## '):
            story.append(Paragraph(clean(stripped[3:]), styles['CustomHeading2']))
        elif stripped.startswith('### '):
            story.append(Paragraph(clean(stripped[4:]), styles['Heading3']))
        elif stripped.startswith(('- ', '* ', '• ')):
            bullet_text = stripped.lstrip('-*• ').strip()
            story.append(Paragraph(f"\u2022 {clean(bullet_text)}", styles['BulletStyle']))
        elif len(stripped) > 1 and stripped[0].isdigit() and '. ' in stripped[:4]:
            story.append(Paragraph(clean(stripped), styles['BulletStyle']))
        else:
            story.append(Paragraph(clean(stripped), styles['CustomBody']))

    doc.build(story)
    return tmp_path


def generate_and_prepare_download(topic):
    """
    Runs report generation, then creates a PDF.
    Returns (report_markdown, pdf_file_path).
    """
    report = asyncio.run(generate_report(topic))  
    pdf_path = markdown_to_pdf(report)             
    return report, pdf_path                        

In [14]:

with gr.Blocks(
    title="Research & Report Generator",
) as demo:

    gr.Markdown(
        """
        # 📝 Research & Report Generator
        **Powered by a multi-agent system**: Researcher → Writer → Editor

        Enter any topic below and the AI agents will:
        1. **Research** the topic thoroughly
        2. **Write** a structured professional report
        3. **Edit** and polish the final output
        """
    )

    with gr.Row():
        topic_input = gr.Textbox(
            label="Research Topic",
            placeholder="e.g., The impact of AI on software development",
            lines=1,
            scale=4
        )
        submit_btn = gr.Button(
            "Generate Report",
            variant="primary",
            scale=1
        )

    report_output = gr.Markdown(
        label="Generated Report",
        value="Your report will appear here...",
        height=600
    )

    
    pdf_download = gr.File(
        label="📥 Download Report as PDF",
        visible=False           
    )

    gr.Markdown(
        "*⏱️ Report generation takes 30-90 seconds depending on topic complexity.*"
    )

    def show_loading():
        return (
            "⏳ **Generating your report...**\n\n"
            "🔍 Step 1: Researching topic...\n\n"
            "✍️ Step 2: Writing report...\n\n"
            "📝 Step 3: Editing & polishing...\n\n"
            "*Please wait, this may take 30-90 seconds...*"
        ), gr.update(visible=False)   

    def run_and_show(topic):
        report, pdf_path = generate_and_prepare_download(topic)
        
        return report, gr.update(value=pdf_path, visible=pdf_path is not None)

    
    submit_btn.click(
        fn=show_loading,
        inputs=None,
        outputs=[report_output, pdf_download]
    ).then(
        fn=run_and_show,
        inputs=[topic_input],
        outputs=[report_output, pdf_download]
    )

    
    topic_input.submit(
        fn=show_loading,
        inputs=None,
        outputs=[report_output, pdf_download]
    ).then(
        fn=run_and_show,
        inputs=[topic_input],
        outputs=[report_output, pdf_download]
    )

In [15]:
if __name__ == "__main__":
    demo.launch()
    demo.launch(share=True)   

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* Running on public URL: https://6e8298ef6b93e1a2d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
